# Combined Holiday Dataset
---

Merges the three cleaned datasets on **country**.



In [ ]:
import pandas as pd

# Load the three cleaned datasets 
cost_of_living = pd.read_csv('Cleaned_Csvs/Cleaned_CostOfLiving.csv')
weather        = pd.read_csv('Cleaned_Csvs/Cleaned_GlobalWeather.csv')
disasters      = pd.read_csv('Cleaned_Csvs/Cleaned_NaturalDisasters.csv')

print(f"Cost of Living rows : {len(cost_of_living)}")
print(f"Weather rows        : {len(weather)}")
print(f"Disasters rows      : {len(disasters)}")

for name, df in [("Cost of Living", cost_of_living), ("Weather", weather), ("Disasters", disasters)]:
    nan_counts = df.isnull().sum()
    nan_counts = nan_counts[nan_counts > 0]
    print(f"\n--- {name} NaN counts ---")
    if nan_counts.empty:
        print("  No NaN values")
    else:
        print(nan_counts.to_string())


Cost of Living rows : 203
Weather rows        : 201
Disasters rows      : 156

--- Cost of Living NaN counts ---
  No NaN values

--- Weather NaN counts ---
  No NaN values

--- Disasters NaN counts ---
  No NaN values


In [ ]:

import re

def normalise_country(s):
    """Lowercase and strip all whitespace for robust country matching."""
    return re.sub(r'\s+', '', str(s).lower().strip())

# Add a normalised key to each dataset
for df in [cost_of_living, weather, disasters]:
    df['_country_key'] = df['country'].apply(normalise_country)

# Step 1: inner join cost_of_living + weather (both MUST be present)
combined = pd.merge(
    cost_of_living.drop(columns=['_country_key']),
    weather.drop(columns=['_country_key']),
    on='country',
    how='inner'
)
combined['_country_key'] = combined['country'].apply(normalise_country)
print(f"After inner join (cost_of_living + weather): {len(combined)} rows")

# Step 2: left join disasters onto combined
#   - countries in combined with a disaster record will be filled with that records data
#   - countries in combined with NO disaster record disaster columns fill 
#     with 0 as I assume no disasters for those countries
#   - disaster countries NOT in combined are dropped 
disaster_numeric_cols = disasters.select_dtypes(include='number').columns.tolist()

combined = pd.merge(
    combined,
    disasters.drop(columns=['_country_key']),
    on='country',
    how='left'
)

# Fill missing disaster numeric values with 0
combined[disaster_numeric_cols] = combined[disaster_numeric_cols].fillna(0)

combined = combined.drop(columns=['_country_key']).sort_values('country').reset_index(drop=True)

print(f"Combined rows : {len(combined)}")
print(f"Combined cols : {combined.shape[1]}")
combined.head(10)


After inner join (cost_of_living + weather): 170 rows
Combined rows : 170
Combined cols : 15


,country,Mid_range_meal_pp,One_way_local_transport_ticket,Apartment,temperature_celsius,condition_text,wind_kph,humidity,uv_index,condition_category,HRS_of_light,Disaster_Count,Median_Severity,Max_Severity,Unique_Days_Under_Threat
0,Afghanistan,3.41,0.22,79.900,26.6,partly cloudy,13.3,24,7.0,Partly Cloudy,0.58,6.0,0.50,1.0,22.0
1,Albania,12.48,0.43,186.790,19.0,partly cloudy,11.2,94,5.0,Partly Cloudy,0.61,1.0,0.00,0.0,7.0
2,Algeria,6.66,0.21,143.880,23.0,sunny,15.1,29,5.0,Clear,0.59,5.0,0.25,1.0,8.0
3,Andorra,20.17,1.90,811.730,6.3,light drizzle,11.9,61,2.0,Rain,0.61,0.0,0.00,0.0,0.0
4,Angola,32.12,0.59,220.390,26.0,partly cloudy,13.0,89,8.0,Partly Cloudy,0.49,2.0,1.00,1.0,11.0
5,Antigua And Barbuda,46.26,1.02,393.905,26.0,partly cloudy,9.0,84,1.0,Partly Cloudy,0.54,0.0,0.00,0.0,0.0
6,Argentina,13.76,0.35,205.320,8.0,clear,3.6,93,1.0,Clear,0.43,4.0,0.50,1.0,62.0
7,Armenia,15.84,0.25,540.260,19.0,partly cloudy,6.8,40,4.0,Partly Cloudy,0.60,2.0,0.50,1.0,6.0
8,Australia,31.15,2.75,1142.780,9.0,clear,4.0,87,1.0,Clear,0.43,8.0,0.25,0.5,27.0
9,Austria,28.49,2.41,707.550,16.0,partly cloudy,20.2,63,5.0,Partly Cloudy,0.64,2.0,0.00,0.0,9.0


In [15]:
combined.to_csv('Combined_Holiday.csv', index=False)
print("Saved to Combined_Holiday.csv")
combined.info()

Saved to Combined_Holiday.csv
<class 'pandas.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         170 non-null    str    
 1   Mid_range_meal_pp               170 non-null    float64
 2   One_way_local_transport_ticket  170 non-null    float64
 3   Apartment                       170 non-null    float64
 4   temperature_celsius             170 non-null    float64
 5   condition_text                  170 non-null    str    
 6   wind_kph                        170 non-null    float64
 7   humidity                        170 non-null    int64  
 8   uv_index                        170 non-null    float64
 9   condition_category              170 non-null    str    
 10  HRS_of_light                    170 non-null    float64
 11  Disaster_Count                  170 non-null    float64
 12  Median_Severity  